# EE-1.1: Circuits from zero

## Objectives

At the end of this notebook you should be able to:

- explain charge, current, voltage and power, and tell **power** (MW) from **energy** (MWh)
- use Ohm's law and explain why transmission grids run at high voltage
- apply Kirchhoff's current and voltage laws
- solve a DC network with **nodal analysis**, by hand and as a matrix equation in NumPy
- validate a solution with KCL and an energy balance

This is the first lesson of the foundations track. You need no electrical background: everything is built from first principles. The matrix you meet at the end of this notebook comes back in Module 04 as the **Y-bus**, the heart of every power flow calculation. And Kirchhoff's current law is the **nodal balance** constraint in every PyPSA model you will build.

In [1]:
import numpy as np
import pandas as pd

### Charge, current, voltage

**Charge** (*q*, coulomb C) is the basic quantity; electrons carry it and can move through a wire.

**Current** (*I*, ampere A) is the rate at which charge flows past a point: *I = dq/dt*, so 1 A = 1 C/s. By convention current flows from + to − (opposite to the electrons — a historical accident).

**Voltage** (*V*, volt V) is energy per unit of charge: 1 V = 1 J/C. It is always a **difference** between two points. "The voltage at node 1" only means something relative to an agreed **reference point** — remember this, because it is why every power flow calculation needs a slack bus.

> The water analogy (voltage ≈ pressure, current ≈ flow rate, resistor ≈ narrow pipe) helps for DC, but it breaks down for AC and reactive power. Don't build on it.

### Power and energy

Multiply energy per charge (V) by charge per second (A) and you get energy per second:

**P = V · I**, in watts (W = J/s).

**Power** (W, kW, MW, GW) is a *rate*. **Energy** (Wh, kWh, MWh, TWh) is power accumulated over time: *E = P · t*. Capacities and instantaneous demand are MW; market prices are €/MWh (per unit of energy); storage size is MWh and storage power is MW. A model that mixes them up still solves — and returns nonsense.

In [2]:
plant_MW = 400
hours = 6
energy_MWh = plant_MW * hours
print(f"A {plant_MW} MW plant running {hours} h produces {energy_MWh:,} MWh")

A 400 MW plant running 6 h produces 2,400 MWh


Notice the unit arithmetic: MW × h = MWh. Writing the units next to every number, as above, catches most mistakes before they happen.

### Ohm's law and why grids use high voltage

A **resistor** turns electrical energy into heat. For many materials the current is proportional to the voltage across them:

**V = I · R** (Ohm's law), resistance *R* in ohms (Ω). Its inverse, **conductance** *G = 1/R* (siemens S), will matter more to us: *I = G · V*.

Substituting into P = V · I gives the power lost as heat: **P_loss = I² · R**. To deliver power *P* at voltage *V*, the current is *I = P/V*, so

**P_loss = (P/V)² · R** — doubling the voltage cuts losses by a factor of four.

In [3]:
def line_loss_MW(power_MW, voltage_kV, resistance_ohm):
    """I²R loss (MW) when delivering power_MW at voltage_kV (DC simplification)."""
    current_A = power_MW * 1e6 / (voltage_kV * 1e3)
    return current_A**2 * resistance_ohm / 1e6

rows = []
for kv in [110, 220, 400]:
    loss = line_loss_MW(100, kv, 10)
    rows.append({"voltage_kV": kv, "current_A": 100e6 / (kv * 1e3),
                 "loss_MW": loss, "loss_share_%": 100 * loss / 100})
pd.DataFrame(rows).round(2)

   voltage_kV  current_A  loss_MW  loss_share_%
0         110     909.09     8.26          8.26
1         220     454.55     2.07          2.07
2         400     250.00     0.62          0.62

Delivering 100 MW over a 10 Ω line loses 8.3 % at 110 kV but only 0.6 % at 400 kV. This is why long-distance transmission runs at 220–400 kV.

> **Simplification:** this treats the line as a DC circuit. Real grids are three-phase AC (EE-2, EE-3), which changes the numbers but not the I²R principle. Naming your simplifications is a habit worth building from day one.

### Kirchhoff's laws

Two conservation laws govern every circuit, from a torch to the European grid:

- **Kirchhoff's Current Law (KCL):** at every node, the currents flowing in equal the currents flowing out — charge cannot pile up. In PyPSA this is the **nodal balance** at each bus: generation − demand ± flows = 0. Its dual variable becomes the electricity price (Module 03).
- **Kirchhoff's Voltage Law (KVL):** around any closed loop, the voltage differences sum to zero — potential is like height. KVL is why power **splits across parallel paths by impedance** instead of going wherever is cheapest; this separates a real network from a simple transport model (Module 04).

### Nodal analysis: a worked example

**Nodal analysis** turns a circuit into equations:

1. Choose a **reference node** and set its voltage to 0 (in power systems: the slack bus).
2. The unknowns are the voltages of all other nodes.
3. Write KCL at each non-reference node, with each branch current = conductance × voltage difference.
4. Solve.

![Two-node DC circuit](images/ee1_two_node_circuit.svg)

A source injects 3 A into node 1. Counting currents *leaving* a node as positive:

- Node 1: (V₁ − 0)/4 + (V₁ − V₂)/2 = 3 → **0.75 V₁ − 0.5 V₂ = 3**
- Node 2: (V₂ − 0)/4 + (V₂ − V₁)/2 = 0 → **−0.5 V₁ + 0.75 V₂ = 0**

From node 2, V₂ = (2/3) V₁. Substituting: (5/12) V₁ = 3, so **V₁ = 7.2 V** and **V₂ = 4.8 V**.

### The same problem as a matrix equation

The two equations form the linear system **G · V = I**. Look at the structure of **G**: the diagonal holds the sum of conductances connected to each node (1/4 + 1/2 = 0.75), the off-diagonals hold minus the conductance between two nodes (−1/2). It is symmetric, and in a large network it would be **sparse**. This is exactly the structure of the Y-bus in Module 04.

In [4]:
G = np.array([[0.75, -0.50],
              [-0.50, 0.75]])   # conductance matrix, S
I = np.array([3.0, 0.0])         # injected currents, A

V = np.linalg.solve(G, I)        # node voltages, V
V

array([7.2, 4.8])

Use `np.linalg.solve`, never `np.linalg.inv(G) @ I`: solving is faster and numerically more stable. Large networks use sparse solvers from `scipy.sparse.linalg`.

### Validate before you trust it

A solution is only finished once it has been checked against physics. Two checks work for every circuit: KCL at every node, and an **energy balance** — the power delivered by the source must equal the power turned into heat.

In [5]:
V1, V2 = V
I_10 = V1 / 4            # A, node 1 -> reference
I_12 = (V1 - V2) / 2     # A, node 1 -> node 2
I_20 = V2 / 4            # A, node 2 -> reference

assert np.isclose(I_10 + I_12, 3.0), "KCL violated at node 1"
assert np.isclose(I_12, I_20), "KCL violated at node 2"

p_source_W = V1 * 3.0
p_heat_W = I_10**2 * 4 + I_12**2 * 2 + I_20**2 * 4
assert np.isclose(p_source_W, p_heat_W), "energy balance violated"

print(f"Branch currents: I10 = {I_10:.2f} A, I12 = {I_12:.2f} A, I20 = {I_20:.2f} A")
print(f"Source power {p_source_W:.2f} W = heat {p_heat_W:.2f} W")

Branch currents: I10 = 1.80 A, I12 = 1.20 A, I20 = 1.20 A
Source power 21.60 W = heat 21.60 W


Both checks pass: 1.8 A + 1.2 A leave node 1, matching the 3 A injected, and the 21.6 W from the source equals the 21.6 W dissipated in the resistors. If these ever disagree, the model is wrong — no matter what the solver says.

## Check your understanding

**Power and energy**

1. Before computing: a battery is rated 50 MW / 200 MWh. How many hours can it discharge at full power? Then check with code.
2. Change the line resistance in `line_loss_MW` to 5 Ω. Predict the 110 kV loss first, then run it.

**Nodal analysis**

1. Change the injection in the worked example from 3 A to 6 A. Predict V₁ and V₂ *without* solving (hint: the system is linear), then verify.
2. Increase R₁₂ from 2 Ω to 20 Ω. Write the new **G**, predict whether V₁ rises or falls, and verify. What happens to the current reaching node 2?

**Thinking ahead**

1. Why does a voltage need a reference point, and what would go wrong in `np.linalg.solve` without one? (The exercises explore this.)
2. Where will you see KCL again when you build your first PyPSA network?